# Descenso de Gradiente — Prototipo

Proyecto: Optimización de algoritmos de Inteligencia Artificial mediante Cálculo Diferencial.

Este notebook implementa el algoritmo de Descenso de Gradiente desde cero (NumPy) y lo compara contra una búsqueda aleatoria de parámetros.

## 1. Generación del dataset

Dataset sintético de clasificación binaria, exportado a `dataset.csv`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=RANDOM_STATE,
)

dataset = pd.DataFrame(X, columns=["x1", "x2"])
dataset["y"] = y
dataset.to_csv("dataset.csv", index=False)

print(f"Dataset generado: {dataset.shape[0]} muestras, {dataset.shape[1] - 1} features")
dataset.head()


## 2. Función de costo (MSE) y su derivada

Implementación explícita del Error Cuadrático Medio y de ∂J/∂w.

In [ ]:
def sigmoid(z):
    """sigma(z) = 1 / (1 + e^-z), mapea la salida lineal a un rango [0, 1]."""
    return 1 / (1 + np.exp(-z))


def predict(X, W, b):
    """y_hat = sigma(X.W + b)"""
    z = X @ W + b
    return sigmoid(z)


def cost_function(X, y, W, b):
    """J(W, b) = (1/n) * sum((y_hat - y)^2)  -- Error Cuadratico Medio."""
    y_hat = predict(X, W, b)
    return np.mean((y_hat - y) ** 2)


def gradient(X, y, W, b):
    """
    Derivada explicita de J respecto a W y b (regla de la cadena):
        dJ/dy_hat = (2/n) * (y_hat - y)
        dy_hat/dz = y_hat * (1 - y_hat)          <- derivada de la sigmoide
        dz/dW = X ; dz/db = 1

        dJ/dW = (2/n) * sum((y_hat - y) * y_hat*(1-y_hat) * x)
        dJ/db = (2/n) * sum((y_hat - y) * y_hat*(1-y_hat))
    """
    n = X.shape[0]
    y_hat = predict(X, W, b)
    delta = (2 / n) * (y_hat - y) * y_hat * (1 - y_hat)
    dW = X.T @ delta
    db = np.sum(delta)
    return dW, db


# Prueba rapida con pesos iniciales en cero
W_test = np.zeros(X.shape[1])
b_test = 0.0
print(f"Costo inicial (W=0, b=0): {cost_function(X, y, W_test, b_test):.4f}")
dW_test, db_test = gradient(X, y, W_test, b_test)
print(f"Gradiente inicial -> dW: {dW_test}, db: {db_test:.4f}")


## 3. Descenso de Gradiente

Bucle iterativo W ← W − α·∂J/∂w, registrando el error en cada iteración.

In [ ]:
ALPHA = 1.0        # tasa de aprendizaje
MAX_ITER = 5000     # tope de iteraciones
TOL = 1e-6          # umbral de convergencia: |J(t) - J(t-1)| < TOL


def gradient_descent(X, y, alpha=ALPHA, max_iter=MAX_ITER, tol=TOL):
    """
    Descenso de Gradiente desde cero:
        W(t+1) = W(t) - alpha * dJ/dW
        b(t+1) = b(t) - alpha * dJ/db
    Se detiene cuando el costo deja de mejorar mas que `tol`, o al llegar a max_iter.
    Devuelve W, b finales y el historial de costo (uno por iteracion).
    """
    W = np.zeros(X.shape[1])
    b = 0.0
    cost_history = [cost_function(X, y, W, b)]

    for i in range(1, max_iter + 1):
        dW, db = gradient(X, y, W, b)
        W = W - alpha * dW
        b = b - alpha * db
        cost_history.append(cost_function(X, y, W, b))

        if abs(cost_history[-2] - cost_history[-1]) < tol:
            break

    return W, b, cost_history


W_gd, b_gd, cost_history_gd = gradient_descent(X, y)
iters_gd = len(cost_history_gd) - 1

y_pred_gd = (predict(X, W_gd, b_gd) >= 0.5).astype(int)
accuracy_gd = np.mean(y_pred_gd == y)

print(f"Descenso de Gradiente -> iteraciones: {iters_gd}")
print(f"Costo inicial: {cost_history_gd[0]:.4f} | Costo final: {cost_history_gd[-1]:.4f}")
print(f"W final: {W_gd}, b final: {b_gd:.4f}")
print(f"Precision final: {accuracy_gd:.4f}")


## 4. Búsqueda Aleatoria (baseline)

Mismo problema resuelto probando parámetros al azar, para comparación.

In [ ]:
W_RANGE = (-10, 10)   # rango de busqueda para cada peso
B_RANGE = (-10, 10)   # rango de busqueda para el bias
MAX_ITER_RS = 5000    # mismo tope que Descenso de Gradiente, para dar el mismo presupuesto de intentos
PATIENCE = 500         # se detiene si el mejor costo no mejora en 500 intentos seguidos


def random_search(X, y, max_iter=MAX_ITER_RS, patience=PATIENCE,
                   w_range=W_RANGE, b_range=B_RANGE, seed=RANDOM_STATE):
    """
    Baseline sin calculo: en cada intento se sortean W y b al azar (uniforme)
    dentro de un rango, se evalua el costo con cost_function, y se conserva
    el mejor par (W, b) encontrado hasta el momento.
    """
    rng = np.random.default_rng(seed)

    best_W = rng.uniform(*w_range, size=X.shape[1])
    best_b = rng.uniform(*b_range)
    best_cost = cost_function(X, y, best_W, best_b)
    cost_history = [best_cost]
    no_improve = 0

    for i in range(1, max_iter + 1):
        W_candidate = rng.uniform(*w_range, size=X.shape[1])
        b_candidate = rng.uniform(*b_range)
        cost_candidate = cost_function(X, y, W_candidate, b_candidate)

        if cost_candidate < best_cost:
            best_W, best_b, best_cost = W_candidate, b_candidate, cost_candidate
            no_improve = 0
        else:
            no_improve += 1

        cost_history.append(best_cost)

        if no_improve >= patience:
            break

    return best_W, best_b, cost_history


W_rs, b_rs, cost_history_rs = random_search(X, y)
iters_rs = len(cost_history_rs) - 1

y_pred_rs = (predict(X, W_rs, b_rs) >= 0.5).astype(int)
accuracy_rs = np.mean(y_pred_rs == y)

print(f"Busqueda Aleatoria -> iteraciones: {iters_rs}")
print(f"Costo inicial: {cost_history_rs[0]:.4f} | Costo final: {cost_history_rs[-1]:.4f}")
print(f"W final: {W_rs}, b final: {b_rs:.4f}")
print(f"Precision final: {accuracy_rs:.4f}")


## 5. Métricas

CPU, RAM y tiempo real (medidos con `psutil`), iteraciones hasta converger, precisión final — para ambos métodos.

In [ ]:
import time
import psutil


def measure(func, *args, **kwargs):
    """
    Ejecuta func(*args, **kwargs) y devuelve (resultado, tiempo_s, cpu_pct, ram_delta_mb).
    cpu_percent() necesita una primera llamada de "calentamiento" antes de medir,
    porque la primera lectura siempre devuelve 0.0 (no tiene punto de referencia previo).
    """
    process = psutil.Process()
    process.cpu_percent(interval=None)  # calentamiento: fija el punto de partida
    mem_before = process.memory_info().rss

    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    t1 = time.perf_counter()

    cpu_pct = process.cpu_percent(interval=None)
    mem_after = process.memory_info().rss

    elapsed = t1 - t0
    ram_delta_mb = (mem_after - mem_before) / (1024 ** 2)
    return result, elapsed, cpu_pct, ram_delta_mb


(W_gd_m, b_gd_m, cost_history_gd_m), time_gd, cpu_gd, ram_gd = measure(gradient_descent, X, y)
iters_gd_m = len(cost_history_gd_m) - 1
accuracy_gd_m = np.mean((predict(X, W_gd_m, b_gd_m) >= 0.5).astype(int) == y)

(W_rs_m, b_rs_m, cost_history_rs_m), time_rs, cpu_rs, ram_rs = measure(random_search, X, y)
iters_rs_m = len(cost_history_rs_m) - 1
accuracy_rs_m = np.mean((predict(X, W_rs_m, b_rs_m) >= 0.5).astype(int) == y)

metricas = pd.DataFrame([
    {
        "Metodo": "Descenso de Gradiente",
        "Iteraciones": iters_gd_m,
        "Tiempo (s)": time_gd,
        "CPU (%)": cpu_gd,
        "RAM (MB)": ram_gd,
        "Costo final": cost_history_gd_m[-1],
        "Precision final": accuracy_gd_m,
    },
    {
        "Metodo": "Busqueda Aleatoria",
        "Iteraciones": iters_rs_m,
        "Tiempo (s)": time_rs,
        "CPU (%)": cpu_rs,
        "RAM (MB)": ram_rs,
        "Costo final": cost_history_rs_m[-1],
        "Precision final": accuracy_rs_m,
    },
])

metricas.to_csv("metricas.csv", index=False)
metricas


## 6. Gráficos

Curva de error vs. iteración y comparación entre métodos, exportados como PNG para la diapositiva 12 del PPT.

In [ ]:
import matplotlib.pyplot as plt

# Grafico 1: curva de error (costo) vs iteracion, para ambos metodos
fig1, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(cost_history_gd, label="Descenso de Gradiente", color="tab:blue")
ax1.plot(cost_history_rs, label="Busqueda Aleatoria", color="tab:orange")
ax1.set_xlabel("Iteracion")
ax1.set_ylabel("Costo (MSE)")
ax1.set_title("Curva de convergencia: Costo vs Iteracion")
ax1.legend()
ax1.grid(alpha=0.3)
fig1.tight_layout()
fig1.savefig("curva_convergencia.png", dpi=150)
plt.show()

# Grafico 2: comparacion de metricas entre metodos (barras)
fig2, axes = plt.subplots(1, 3, figsize=(12, 4))
metodos = metricas["Metodo"]
colores = ["tab:blue", "tab:orange"]

axes[0].bar(metodos, metricas["Iteraciones"], color=colores)
axes[0].set_title("Iteraciones hasta converger")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(metodos, metricas["Tiempo (s)"], color=colores)
axes[1].set_title("Tiempo de ejecucion (s)")
axes[1].tick_params(axis="x", rotation=15)

axes[2].bar(metodos, metricas["Precision final"], color=colores)
axes[2].set_title("Precision final")
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis="x", rotation=15)

fig2.suptitle("Comparacion Descenso de Gradiente vs Busqueda Aleatoria")
fig2.tight_layout()
fig2.savefig("comparacion_metodos.png", dpi=150)
plt.show()

# Grafico 3: frontera de decision del modelo entrenado con Descenso de Gradiente
fig3, ax3 = plt.subplots(figsize=(6, 6))

x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))
grid = np.c_[xx1.ravel(), xx2.ravel()]
zz = predict(grid, W_gd, b_gd).reshape(xx1.shape)

# colores del fondo alineados con la paleta 'coolwarm' de los puntos:
# celeste = region de baja probabilidad (prediccion clase 0, igual que los puntos azules)
# rosado  = region de alta probabilidad (prediccion clase 1, igual que los puntos rojos)
ax3.contourf(xx1, xx2, zz, levels=[0, 0.5, 1], colors=["#deebf7", "#fde0dd"], alpha=0.6)
ax3.contour(xx1, xx2, zz, levels=[0.5], colors="black", linewidths=1.5)
ax3.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=25)
ax3.set_xlabel("x1")
ax3.set_ylabel("x2")
ax3.set_title("Frontera de decision (Descenso de Gradiente)")
fig3.tight_layout()
fig3.savefig("frontera_decision.png", dpi=150)
plt.show()

print("Graficos exportados: curva_convergencia.png, comparacion_metodos.png, frontera_decision.png")
